In [17]:
import tarfile
import pandas as pd
import os
import numpy as np
import spacy
import math
from bow_pipeline import BOWpipe
from bow_pipeline import Gaussian_Classifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score
from nltk.util import defaultdict

ImportError: cannot import name 'Gaussian_Classifier' from 'bow_pipeline' (/Users/kahncant/Documents/NLP/Lab 2/bow_pipeline.py)

In [4]:
def load_dataset(split='train'):
    data = []
    sentiment = {"pos": 1, "neg": 0}
    split_path = os.path.join('./aclImdb/', split)
    for label in ["pos", "neg"]:
        label_path = os.path.join(split_path, label)
        for fname in os.listdir(label_path):
            if fname.endswith(".txt"):
                with open(os.path.join(label_path, fname), encoding="utf-8") as f:
                    text = f.read().strip()
                    data.append([sentiment[label], text])
    return pd.DataFrame(data, columns=['label', 'text']).sample(500)

print("Loading training data...")
dataset = load_dataset(split="train")
print(f"Loaded {len(dataset)} examples.")

Loading training data...
Loaded 500 examples.


In [5]:
print("Example of a positive review:")
print(dataset[dataset['label'] == 1].iloc[50]['text'])
print("Example of a negative review:")
print(dataset[dataset['label'] == 0].iloc[50]['text'])
print(dataset.shape)

Example of a positive review:
*Possible Spoilers* Although done before (and better) in 'Midnight Express' and 'Return To Paradise', Brokedown Palace still strikes a chord with me.<br /><br />Here we have the tale of two young girls who travel through Thailand, and get arrested on charges of trafficking drugs. Was it Clare Danes' Alice? Was it Kate Beckinsale's Darlene? Was it the handsome stranger whom they met on their journey? None of that really matters, for this is a tale of friendship and trust and the limits they can be stretched to. Throw in Bill Pullman as an unenthusiastic lawyer and Jacqueline Kim as his Thai bride (and better lawyer than he is) and we have a nice little story that holds the audience's attention.<br /><br />Brokedown Palace is nothing extraordinary, or notorious for any reason - it is not an original concept, it doesn't show sensationalised violence that leads to the wannabe-avant-garde crowd talking about it's "gritty realism" or "hard hitting truths" - it i

In [6]:
print("Encoding Unigram case...")
ug_raw = BOWpipe(dataset['text'])
print("Encoding Trigram case...")
tg_raw = BOWpipe(dataset['text'], n=3)
print("Encoding Unigram TF-IDF case...")
ug_tfidf = BOWpipe(dataset['text'], tfidf=True)
print("Encoding Trigram TF-IDF case...")
tg_tfidf = BOWpipe(dataset['text'], n=3, tfidf=True)

Encoding Unigram case...
Encoding Trigram case...
Encoding Unigram TF-IDF case...
Encoding Trigram TF-IDF case...


In [7]:
def convert_size(size_bytes):
   if size_bytes == 0:
       return "0B"
   size_name = ("B", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB")
   i = int(math.floor(math.log(size_bytes, 1024)))
   p = math.pow(1024, i)
   s = round(size_bytes / p, 2)
   return "%s %s" % (s, size_name[i])

print(f"Size of the Unigram feature set: {ug_raw.shape[0]}x{ug_raw.shape[1]} which takes up {convert_size(ug_raw.nbytes)}.")
print(f"Size of the Unigram TF-IDF feature set: {ug_tfidf.shape[0]}x{ug_tfidf.shape[1]} which takes up {convert_size(ug_tfidf.nbytes)}.")
print(f"Size of the Trigram feature set: {tg_raw.shape[0]}x{tg_raw.shape[1]} which takes up {convert_size(tg_raw.nbytes)}.")
print(f"Size of the Trigram TF-IDF feature set: {tg_tfidf.shape[0]}x{tg_tfidf.shape[1]} which takes up {convert_size(tg_tfidf.nbytes)}.")

Size of the Unigram feature set: 500x14182 which takes up 54.1 MB.
Size of the Unigram TF-IDF feature set: 500x14182 which takes up 54.1 MB.
Size of the Trigram feature set: 500x66767 which takes up 254.7 MB.
Size of the Trigram TF-IDF feature set: 500x66767 which takes up 254.7 MB.


In [9]:
ug = Gaussian_Classifier("Unigram", ug_raw.encoding, dataset['label'])
ug.evaluate_model()
ug_tfidf = Gaussian_Classifier("Unigram TF-IDF", ug_tfidf.encoding, dataset['label'])
ug_tfidf.evaluate_model()
tg = Gaussian_Classifier("Trigram", tg_raw.encoding, dataset['label'])
tg.evaluate_model()
tg_tfidf = Gaussian_Classifier("Trigram TF-IDF", tg_tfidf.encoding, dataset['label'])
tg_tfidf.evaluate_model()

Training Unigram model...
Unigram: Acc. - 0.68%, Prec. - 0.73%, Rec. - 0.68%, 
Training Unigram TF-IDF model...
Unigram TF-IDF: Acc. - 0.67%, Prec. - 0.71%, Rec. - 0.69%, 
Training Trigram model...
Trigram: Acc. - 0.55%, Prec. - 0.66%, Rec. - 0.43%, 
Training Trigram TF-IDF model...
Trigram TF-IDF: Acc. - 0.55%, Prec. - 0.65%, Rec. - 0.44%, 
